# Notebook 2: Controlled training runs (resolution and model comparison)
EAAI-26-20775 revision, Days 1–2.

Every run uses **exactly the same data, split and training settings as your baseline** (read from its `args.yaml`).
Only the model and/or training input size changes, so differences come from that change alone.

**Runs** (pick one in the Settings cell, run all, wait about 1.5 h, then pick the next):
| RUN | Model | Train size | Answers |
|---|---|---|---|
| `v8n_640` | YOLOv8n | 640 | Reviewer 2.6 (640 train / 640 infer) |
| `v8n_768_rerun` | YOLOv8n | 768 | Same as baseline, retrained now; shows run-to-run variation and gives a same-version reference for the comparison |
| `v10n_768` | YOLOv10n | 768 | Reviewers 1.4, 2.5 |
| `v11n_768` | YOLO11n | 768 | Reviewers 1.4, 2.5 |

Checkpoints are saved to Drive every epoch. If Colab disconnects, reconnect and **Run all** again; training resumes where it stopped.

**Before running:** Runtime → Change runtime type → T4 GPU.

In [ ]:
# ---- Settings ----
RUN = "v8n_640"          # one of: v8n_640, v8n_768_rerun, v10n_768, v11n_768
DRIVE_ROOT = "/content/drive/MyDrive/fabric"
DATASET_ZIP = f"{DRIVE_ROOT}/dataset_clean.zip"
BASELINE_RUN = f"{DRIVE_ROOT}/runs_clean/detect/fabric_defect_yolov8n_clean"
OUT_DIR = f"{DRIVE_ROOT}/revision/runs"
WORK = "/content/data"
DEVICE = 0
LOCAL_TEST = False       # leave False on Colab
EPOCHS_OVERRIDE = None   # leave None (uses baseline epochs)

RUNS = {
    "v8n_640":       {"model": "yolov8n.pt", "imgsz": 640},
    "v8n_768_rerun": {"model": "yolov8n.pt", "imgsz": 768},
    "v10n_768":      {"model": "yolov10n.pt", "imgsz": 768},
    "v11n_768":      {"model": "yolo11n.pt", "imgsz": 768},
}
assert RUN in RUNS, f"RUN must be one of {list(RUNS)}"

In [ ]:
import os, sys, subprocess
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---- Dataset ----
import zipfile, glob, yaml
if not glob.glob(f"{WORK}/**/images/train", recursive=True):
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(WORK)
root = os.path.dirname(os.path.dirname(sorted(glob.glob(f"{WORK}/**/images/train", recursive=True))[0]))
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
counts = {s: len([f for f in os.listdir(f"{root}/images/{s}") if f.lower().endswith(IMG_EXT)]) for s in ("train", "val")}
print("Dataset:", root, counts)
base_args = yaml.safe_load(open(f"{BASELINE_RUN}/args.yaml"))
from ultralytics import YOLO as _Y   # class order taken from the baseline model, so all runs match it
names = {int(k): v for k, v in _Y(f"{BASELINE_RUN}/weights/best.pt").names.items()}
data_yaml = f"{WORK}/data.yaml"
yaml.safe_dump({"path": root, "train": "images/train", "val": "images/val", "names": names}, open(data_yaml, "w"), sort_keys=False)
print("Classes:", names)

In [ ]:
# ---- Build training arguments = baseline args.yaml, with only model / imgsz / paths changed ----
import ultralytics, json
from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG_DICT
cfg = RUNS[RUN]
SKIP = {"model", "data", "project", "name", "exist_ok", "resume", "mode", "task", "save_dir", "device", "imgsz", "source", "pretrained"}
train_args = {k: v for k, v in base_args.items() if k in DEFAULT_CFG_DICT and k not in SKIP}
dropped = sorted(k for k in base_args if k not in DEFAULT_CFG_DICT and k not in SKIP)
if EPOCHS_OVERRIDE:
    train_args["epochs"] = EPOCHS_OVERRIDE
train_args.update(data=data_yaml, imgsz=cfg["imgsz"], device=DEVICE, project=OUT_DIR, name=RUN, exist_ok=True,
                  pretrained=True)
print("Ultralytics", ultralytics.__version__, "| run:", RUN, cfg)
print("Settings copied from baseline:", {k: train_args.get(k) for k in ("epochs", "batch", "optimizer", "lr0", "momentum",
      "weight_decay", "cos_lr", "patience", "seed", "deterministic", "amp", "mosaic", "mixup", "fliplr", "flipud", "hsv_h", "hsv_s", "hsv_v")})
if dropped:
    print("Baseline settings not recognised by this Ultralytics version (ignored):", dropped)
run_dir = f"{OUT_DIR}/{RUN}"
os.makedirs(run_dir, exist_ok=True)
json.dump({"run": RUN, "cfg": cfg, "ultralytics": ultralytics.__version__, "train_args": train_args, "dropped": dropped},
          open(f"{run_dir}/run_config.json", "w"), indent=2, default=str)

In [ ]:
# ---- Train (resumes automatically after a disconnect) ----
last = f"{run_dir}/weights/last.pt"
best = f"{run_dir}/weights/best.pt"
done_flag = f"{run_dir}/TRAINING_DONE"
if os.path.exists(done_flag):
    print("Training already finished for", RUN)
elif os.path.exists(last):
    print("Resuming from", last)
    YOLO(last).train(resume=True)
else:
    YOLO(cfg["model"]).train(**train_args)
open(done_flag, "w").write("done")

In [ ]:
# ---- Evaluate best.pt at 640 (deployment size) and at its training size ----
model = YOLO(best)
names = {int(k): v for k, v in model.names.items()}
def metrics_dict(r):
    b = r.box
    out = {"mAP50": float(b.map50), "mAP50_95": float(b.map), "P": float(b.mp), "R": float(b.mr), "per_class": {}}
    for i, c in enumerate(b.ap_class_index):
        p, rcl, ap50, ap = b.class_result(i)
        out["per_class"][names[int(c)]] = {"P": float(p), "R": float(rcl), "AP50": float(ap50), "AP50_95": float(ap)}
    return out
evals = {}
for sz in sorted({640, cfg["imgsz"]}):
    r = model.val(data=data_yaml, imgsz=sz, batch=16, device=DEVICE, split="val", plots=True,
                  project=run_dir, name=f"val_{sz}", exist_ok=True, verbose=False)
    evals[f"infer{sz}"] = metrics_dict(r)
    m = evals[f"infer{sz}"]
    print(f"\n== {RUN}: evaluated at {sz} ==  mAP50 {m['mAP50']:.3f}  mAP50-95 {m['mAP50_95']:.3f}  P {m['P']:.3f}  R {m['R']:.3f}")
    for c, v in m["per_class"].items():
        print(f"   {c:15s} P {v['P']:.3f}  R {v['R']:.3f}  AP50 {v['AP50']:.3f}")

In [ ]:
# ---- Speed benchmark: same protocol as the manuscript (300 val images, 640, batch 1, 20 warm-up) ----
import numpy as np
val_imgs = sorted(glob.glob(f"{root}/images/val/*"))
for _ in range(20):
    model.predict(val_imgs[0], imgsz=640, device=DEVICE, verbose=False)
pre, inf, post = [], [], []
for im in val_imgs:
    s = model.predict(im, imgsz=640, device=DEVICE, verbose=False)[0].speed
    pre.append(s["preprocess"]); inf.append(s["inference"]); post.append(s["postprocess"])
pre, inf, post = map(np.array, (pre, inf, post)); tot = pre + inf + post
speed = {"pre_ms": [pre.mean(), pre.std()], "inf_ms": [inf.mean(), inf.std()], "post_ms": [post.mean(), post.std()],
         "total_ms": [tot.mean(), tot.std()], "fps_end_to_end": 1000 / tot.mean(), "gpu": (__import__("torch").cuda.get_device_name(0) if not LOCAL_TEST else "cpu")}
n_params = sum(p.numel() for p in model.model.parameters())
speed = {k: ([round(float(x), 3) for x in v] if isinstance(v, list) else (round(float(v), 2) if isinstance(v, float) else v)) for k, v in speed.items()}
print(f"\nSpeed (640, batch 1): total {speed['total_ms'][0]} ± {speed['total_ms'][1]} ms, {speed['fps_end_to_end']} fps on {speed['gpu']}; params {n_params/1e6:.2f} M")

summary = {"run": RUN, "cfg": cfg, "ultralytics": ultralytics.__version__, "params_M": round(n_params / 1e6, 3), "eval": evals, "speed": speed}
json.dump(summary, open(f"{run_dir}/summary.json", "w"), indent=2)
print("\n=== PASTE EVERYTHING BELOW INTO THE CHAT ===")
print(json.dumps(summary, separators=(",", ":")))